<a href="https://colab.research.google.com/github/kavyasri-bhagya/Resume-screening-model-ML/blob/main/resume_screening.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import joblib
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving resume.csv to resume.csv


In [ ]:
df = pd.read_csv(
    "resume.csv",
    encoding="latin1"
)
print(df.columns.tolist())
print(df.shape)
df.head()

['name', 'resume_text', 'job_description', 'match_label']
(500, 4)


,name,resume_text,job_description,match_label
0,Kritika Shah,Name: Kritika Shah || Email: kritika.shah55@gm...,Job Title: FP&A Analyst || Company: Amazon | L...,partial match
1,Arjun Verma,Name: Arjun Verma || Email: arjun.verma70@gmai...,Job Title: Design Lead || Company: Paytm | Loc...,no match
2,Manish Bhatt,Name: Manish Bhatt || Email: manish.bhatt89@gm...,Job Title: Security Analyst || Company: KPMG |...,partial match
3,Natasha Sharma,Name: Natasha Sharma || Email: natasha.sharma7...,Job Title: Sales Director || Company: Ola | Lo...,no match
4,Kavya Agarwal,Name: Kavya Agarwal || Email: kavya.agarwal75@...,Job Title: Business Development Manager || Com...,partial match


In [ ]:
label_mapping = {
    "match": 1,
    "partial match": 1,
    "no match": 0
}
df["label"] = df["match_label"].map(label_mapping)
df[["match_label", "label"]].head()

,match_label,label
0,partial match,1
1,no match,0
2,partial match,1
3,no match,0
4,partial match,1


In [ ]:
def extract_skills(text):
    text = str(text)
    pattern = r"\|\|\s*SKILLS\s*\|\|\s*(.*?)(?=\|\||$)"
    matches = re.findall(
        pattern,
        text,
        flags=re.IGNORECASE | re.DOTALL
    )
    skills = []
    for match in matches:
        extracted_skills = re.split(
            r",|\n",
            match
        )
        for skill in extracted_skills:
            skill = skill.strip()
            if skill:
                skills.append(
                    skill.lower()
                )
    return list(set(skills))
#test
text = "resume: || SKILLS || CFA, Bloomberg Terminal, Equity Research, CA, Valuation, Growth Hacking, Product Analytics ||woked at chennai"
print(extract_skills(text))

['ca', 'product analytics', 'valuation', 'bloomberg terminal', 'equity research', 'cfa', 'growth hacking']


In [ ]:
def extract_experience(text):
    text = str(text).lower()
    patterns = [
        r"(\d+)\+?\s*years",
        r"(\d+)\+?\s*yrs",
        r"(\d+)\+?\s*year",
        r"(\d+)\+?\s*yr"
    ]
    experience_values = []
    for pattern in patterns:
        matches = re.findall(
            pattern,
            text
        )
        for match in matches:
            experience_values.append(
                int(match)
            )
    if experience_values:
        return max(experience_values)
    return 0
#test
sample_text = """
Python Developer
5 years experience in Python
3 years experience in SQL
"""
print(extract_experience(sample_text))

5


In [ ]:
EDUCATION_MAPPING = {
    "phd": 4,
    "doctorate": 4,
    "masters": 3,
    "master": 3,
    "mba": 3,
    "msc": 3,
    "mtech": 3,
    "bachelor": 2,
    "bachelors": 2,
    "btech": 2,
    "be": 2,
    "bsc": 2,
    "bca": 2,
    "diploma": 1
}
def extract_education_score(text):
    text = str(text).lower()
    scores = []
    for degree, score in EDUCATION_MAPPING.items():
        pattern = r"\b" + re.escape(degree) + r"\b"
        if re.search(pattern, text):
            scores.append(score)
    if scores:
        return max(scores)
    return 0
#test
sample_text = """
Completed Bachelor of Technology
and later completed MTech
"""
print(extract_education_score(sample_text))

3


In [ ]:
def generate_features(resume_text, job_description):
    vectorizer = TfidfVectorizer(
        stop_words="english"
    )

    vectors = vectorizer.fit_transform(
        [
            str(resume_text),
            str(job_description)
        ]
    )

    similarity = cosine_similarity(
        vectors[0],
        vectors[1]
    )[0][0]

    resume_skills = set(extract_skills(resume_text))
    jd_skills = set(extract_skills(job_description))

    skill_overlap = len(resume_skills.intersection(jd_skills))

    resume_experience = extract_experience(resume_text)
    jd_experience = extract_experience(job_description)

    experience_match = int(
        resume_experience >= jd_experience
    )

    education_score = extract_education_score(resume_text)

    return [
        similarity,
        skill_overlap,
        resume_experience,
        jd_experience,
        experience_match,
        education_score
    ]

In [ ]:
resume_text = """
Data Analyst
|| SKILLS || Python, SQL, Power BI, Tableau ||
3 years experience
Bachelor Degree
"""
job_description = """
Looking for Data Analyst
Skills:
Python
SQL
Power BI
2 years experience
Bachelor Degree
"""
features = generate_features(
    resume_text,
    job_description
)

print("Similarity =", features[0])
print("Skill Overlap =", features[1])
print("Resume Experience =", features[2])
print("JD Experience =", features[3])
print("Experience Match =", features[4])
print("Education Score =", features[5])

Similarity = 0.8477624970048978
Skill Overlap = 0
Resume Experience = 3
JD Experience = 2
Experience Match = 1
Education Score = 2


In [ ]:
X = []

for index, row in df.iterrows():
    features = generate_features(
        row["resume_text"],
        row["job_description"]
    )

    X.append(features)

y = df["label"]

print("X Length =", len(X))
print("y Length =", len(y))

X Length = 500
y Length = 500


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
print("Training Rows:", len(X_train))
print("Testing Rows:", len(X_test))

Training Rows: 400
Testing Rows: 100


In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
model.fit(X_train,y_train)
print("Random Forest Trained Successfully")

Random Forest Trained Successfully


In [ ]:
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test,predictions)
print("Accuracy:", round(accuracy, 4))
print("\nClassification Report:\n")
print(classification_report(y_test,predictions))

Accuracy: 0.8

Classification Report:

              precision    recall  f1-score   support

           0       0.68      0.89      0.77        38
           1       0.92      0.74      0.82        62

    accuracy                           0.80       100
   macro avg       0.80      0.82      0.80       100
weighted avg       0.83      0.80      0.80       100



In [ ]:
joblib.dump(model,"random_forest.pkl")
print("Model Saved Successfully")

Model Saved Successfully


In [ ]:
new_job_description = """
Looking for Data Analyst

Skills:
SQL
Python
Power BI
Tableau

3 years experience

Bachelor Degree
"""

In [ ]:
candidate_scores = []
for index, row in df.iterrows():
    features = generate_features(
        row["resume_text"],
        new_job_description
    )
    score = model.predict_proba(
        [features]
    )[0][1]
    candidate_scores.append(
        (
            row["name"],
            score
        )
    )
print("Scoring Completed")

Scoring Completed


In [ ]:
top_candidates = sorted(
    candidate_scores,
    key=lambda x: x[1],
    reverse=True
)
print("Total Candidates Ranked:", len(top_candidates))

Total Candidates Ranked: 500


In [ ]:
for rank, candidate in enumerate(top_candidates[:5], start=1):
    print(
        f"{rank}. {candidate[0]} ({candidate[1] * 100:.2f}%)"
    )

1. Kabir Kumar (95.00%)
2. Siddharth Joshi (92.00%)
3. Gaurav Malhotra (92.00%)
4. Kritika Das (91.00%)
5. Suresh Patel (91.00%)
